# Vježba: hoće li se mašina pokvariti?

Isti postupak kao s korisnicima telekom operatera, samo na drugim podacima: **senzori na CNC mašini** (AI4I 2020, UCI).
10.000 mjerenja, a za svako znamo temperature, brzinu okretaja, moment, habanje alata i da li je došlo do kvara.

Ćelije s `# TODO` popunjavate vi. Gdje piše `___`, tu nešto nedostaje.

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

df = pd.read_csv("ai4i2020.csv")
df.head()

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


## 1. Koliko često se mašina pokvari?

In [25]:
# TODO: procenat kvarova
df.columns = df.columns.str.lower().str.replace(" ", "_").str.replace("[", "").str.replace("]", "")
tekstualne = df.select_dtypes(exclude="number").columns
for c in tekstualne:
    df[c] = df[c].str.lower().str.replace(" ", "_")

print(f"Kvarova: {df.machine_failure.sum() * 100 / len(df):.1f} %")
df.head().T

Kvarova: 3.4 %


,0,1,2,3,4
udi,1,2,3,4,5
product_id,m14860,l47181,l47182,l47183,l47184
type,m,l,l,l,l
air_temperature__k,298.1,298.2,298.1,298.2,298.2
process_temperature__k,308.6,308.7,308.5,308.6,308.7
rotational_speed__rpm,1551,1408,1498,1433,1408
torque__nm,42.8,46.3,49.4,39.5,40.0
tool_wear__min,0,3,5,7,9
machine_failure,0,0,0,0,0
twf,0,0,0,0,0


## 2. Train, validacija, test (60 / 20 / 20)

`stratify` čuva isti procenat kvarova u svakom skupu, jer ih je malo.
Kolone `kvar`, `vrsta_kvara` i `id` ne idu u ulaze. Zašto `vrsta_kvara`?

Odgovor: Kvar je ono što model predviđa, što znači da ukoliko uključimo tu kolonu u ulaz, na mjestima gdje ima vrijednost za vrstu kvara, ona bi modelu direktno govorila odgovor i to je curenje podataka. 
Kad želimo predvidjeti kvar, vrsta kvara još ne postoji, obzirom da je ona sama posljedica kvara, a ne i uzrok.

In [28]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1, stratify=df.machine_failure)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1, stratify=df_full_train.machine_failure)

y_train = df_train.machine_failure.values
y_val = df_val.machine_failure.values

kolone = ["type", "air_temperature__k", "process_temperature__k", "rotational_speed__rpm", "torque__nm", "tool_wear__min"]
len(df_train), len(df_val), len(df_test)

(6000, 2000, 2000)

## 3. One-hot encoding (kolona `tip`: L, M, H)

In [29]:
# TODO: fit_transform na train, samo transform na validaciji
dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(df_train[kolone].to_dict(orient="records"))
X_val = dv.transform(df_val[kolone].to_dict(orient="records"))
dv.get_feature_names_out()

array(['air_temperature__k', 'process_temperature__k',
       'rotational_speed__rpm', 'tool_wear__min', 'torque__nm', 'type=h',
       'type=l', 'type=m'], dtype=object)

## 4. Logistička regresija i evaluacija

In [30]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
y_pred = model.predict_proba(X_val)[:, 1]
odluka = (y_pred >= 0.5)

print(f"Accuracy: {accuracy_score(y_val, odluka) * 100:.1f} %")
print(f"Model 'nikad kvar': {(y_val == 0).mean() * 100:.1f} %")

Accuracy: 97.1 %
Model 'nikad kvar': 96.6 %


In [31]:
# TODO: matrica konfuzije, precision i recall
tp = (odluka & (y_val == 1)).sum()
fp = (odluka & (y_val == 0)).sum()
fn = (~odluka & (y_val == 1)).sum()
tn = (~odluka & (y_val == 0)).sum()
print("TN", tn, " FP", fp, " FN", fn, " TP", tp)
print(f"Precision: {tp / (tp + fp):.2f}   Recall: {tp / (tp + fn):.2f}")

TN 1928  FP 4  FN 54  TP 14
Precision: 0.78   Recall: 0.21


**Pitanje:** accuracy je oko 97 %. Je li ovo dobar model? Koliko kvarova od 68 je uhvatio?

Ne, ovo nije dobar model jer od 68 kvarova je samo 14 uhvatio, a 54 propustio. Accuracy je visok zato što je većina mašina već ispravno.

## 5. Probajte druge klasifikatore

Svi imaju isti `fit` i `predict`. Poredimo ih po **F1** (spaja precision i recall).
- **Stablo odlučivanja:** postavlja pitanja da/ne (je li moment > 60?)
- **Random forest:** 100 stabala glasa
- **KNN:** gleda 5 najsličnijih mjerenja iz train skupa
- **Gradient boosting:** stabla se grade jedno za drugim, svako popravlja greške prethodnog

In [32]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

modeli = {
    "logistička regresija": LogisticRegression(max_iter=1000),
    "stablo odlučivanja": DecisionTreeClassifier(max_depth=5, random_state=1),
    "random forest": RandomForestClassifier(n_estimators=100, random_state=1),
    "KNN": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
    "gradient boosting": GradientBoostingClassifier(random_state=1),
}

def uporedi(X_tr, X_v):
    for naziv, m in modeli.items():
        m.fit(X_tr, y_train)
        p = m.predict(X_v)
        print(f"{naziv:22s} accuracy {accuracy_score(y_val, p):.3f}   precision {precision_score(y_val, p, zero_division=0):.2f}   "
              f"recall {recall_score(y_val, p):.2f}   F1 {f1_score(y_val, p):.2f}   uhvaćeno {((p == 1) & (y_val == 1)).sum()}/68")

uporedi(X_train, X_val)

logistička regresija   accuracy 0.971   precision 0.78   recall 0.21   F1 0.33   uhvaćeno 14/68
stablo odlučivanja     accuracy 0.974   precision 0.64   recall 0.51   F1 0.57   uhvaćeno 35/68
random forest          accuracy 0.981   precision 0.97   recall 0.44   F1 0.61   uhvaćeno 30/68
KNN                    accuracy 0.970   precision 0.70   recall 0.21   F1 0.32   uhvaćeno 14/68
gradient boosting      accuracy 0.984   precision 0.95   recall 0.54   F1 0.69   uhvaćeno 37/68


**Pitanje:** koji model je najbolji po F1? A po accuracy? Je li to isti model?

Po F1 je najbolji gradient boosting (0.69), a po accuracy je također gradient boosting (0.984), dakle to je isti model.

## 6. Nove kolone iz fizike

Snaga P = M · ω (ω = 2π · n / 60), razlika temperatura, i habanje × moment.

In [33]:
# TODO: napravite nove kolone
for d in [df_train, df_val]:
    d["snaga_W"] = d.torque__nm * d.rotational_speed__rpm * 2 * np.pi / 60
    d["razlika_temp_C"] = (d.air_temperature__k -273.15) - (d.process_temperature__k - 273.15)
    d["habanje_x_moment"] = d.tool_wear__min * d.torque__nm

kolone_nove = kolone + ["snaga_W", "razlika_temp_C", "habanje_x_moment"]
dv = DictVectorizer(sparse=False)
X_train2 = dv.fit_transform(df_train[kolone_nove].to_dict(orient="records"))
X_val2 = dv.transform(df_val[kolone_nove].to_dict(orient="records"))

uporedi(X_train2, X_val2)

logistička regresija   accuracy 0.971   precision 0.69   recall 0.26   F1 0.38   uhvaćeno 18/68
stablo odlučivanja     accuracy 0.992   precision 0.96   recall 0.78   F1 0.86   uhvaćeno 53/68
random forest          accuracy 0.992   precision 0.98   recall 0.76   F1 0.86   uhvaćeno 52/68
KNN                    accuracy 0.974   precision 0.79   recall 0.32   F1 0.46   uhvaćeno 22/68
gradient boosting      accuracy 0.990   precision 0.94   recall 0.75   F1 0.84   uhvaćeno 51/68


**Pitanje:** šta je više pomoglo, drugi model ili nove kolone?

Više su pomogle nove kolone nego drugi model. Kolone iz fizike (snaga, razlika temperatura, habanje × moment) modelu direktno daju veličine od kojih zavisi kvar.